## KMeans Clustering — Weather Condition Buckets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

PLOTS_DIR = 'presentation_plots'
os.makedirs(PLOTS_DIR, exist_ok=True)
print(f'Plots will be saved to: {PLOTS_DIR}/')

## Load & Clean Data

Reproducing the same cleaning pipeline used across notebooks.

In [ ]:
# This code was given by Simon Clarke!!
# sep=',|\t' is another way to split the clusted data that were not splited by ',' as they should be (csv = comma-seperated values)
# on_bad_lines='warn' is just to signal a warning that there are bad lines in the data (it will just 'warn' us, not 'skip' like what we did)

raw_df = pd.read_csv('Data/Melbourne01.csv',names=["Year", "Month", "Day", "Hour", "Minute",
                                          "Air Temp (degrees C)",
                                          "Apparent Temp (degrees C)", "Dew Pt Temp (degrees C)", "Humidity (%)",
                                          "Wind Direction", "Wind Speed (km/h)", "Wind Gust  (km/h)",
                                          "MSLP (hPa)",
                                          "Rainfall since 9 am (mm)", "Dummy1", "Dummy2"],
                sep=',|\t', on_bad_lines='warn')

In [ ]:
df = raw_df.drop_duplicates()
df = df.reset_index(drop=True)

# Handle shifted rows (missing Year = data shifted one column right)
shifted_rows_index = df.index[df['Year'].isna()].tolist()
df_fixing = df.loc[shifted_rows_index]
df_fixing = df_fixing.drop(columns=['Year'])

col_names=['Year', "Month", "Day", "Hour", "Minute", "Air Temp (degrees C)", "Apparent Temp (degrees C)",
           "Dew Pt Temp (degrees C)", "Humidity (%)", "Wind Direction", "Wind Speed (km/h)", "Wind Gust  (km/h)",
            "MSLP (hPa)", "Rainfall since 9 am (mm)", "Dummy1"]
df_fixing.columns = col_names

df.drop(df.index[shifted_rows_index], inplace=True)
all_df = pd.concat([df, df_fixing])
all_df = all_df.sort_index()

# Drop unnecessary columns
all_df.drop(columns=['Dummy1', 'Dummy2', 'Apparent Temp (degrees C)', 'Wind Gust  (km/h)'], inplace=True)

# Build Timestamp index
all_df['Timestamp'] = pd.to_datetime(all_df[['Year','Month','Day','Hour','Minute']])
all_df.index = all_df['Timestamp']
all_df.drop(columns=['Year','Month','Day','Hour','Minute','Timestamp'], inplace=True)
all_df = all_df.drop_duplicates()

# Fix dtypes
for col in ['Air Temp (degrees C)', 'Dew Pt Temp (degrees C)', 'Humidity (%)',
            'Wind Speed (km/h)', 'MSLP (hPa)', 'Rainfall since 9 am (mm)']:
    all_df[col] = pd.to_numeric(all_df[col], errors='coerce')
all_df['Wind Direction'] = all_df['Wind Direction'].astype(str).str.strip()

print(all_df.shape)
all_df.head()

In [ ]:
# Fix MSLP-shifted rows (Wind Direction holds numeric values instead of compass)
valid_direction = ['N','NNE','NE','ENE','E','ESE','SE','SSE',
                   'S','SSW','SW','WSW','W','WNW','NW','NNW', 'CALM', '-']

MSLP_not_shifted_df = all_df[all_df['Wind Direction'].isin(valid_direction)]
MSLP_shifted_df     = all_df[~all_df['Wind Direction'].isin(valid_direction)]

new_col = ["Air Temp (degrees C)", "Dew Pt Temp (degrees C)", "Humidity (%)",
           "MSLP (hPa)", "Rainfall since 9 am (mm)", "Dummy1", "Dummy2"]
MSLP_shifted_df = MSLP_shifted_df.copy()
MSLP_shifted_df.columns = new_col
MSLP_shifted_df.drop(columns=['Dummy1','Dummy2'], inplace=True)
MSLP_shifted_df['Wind Direction'] = '-'
MSLP_shifted_df['Wind Speed (km/h)'] = -9999.00

all_df = pd.concat([MSLP_not_shifted_df, MSLP_shifted_df])
all_df = all_df.sort_index().drop_duplicates()

# Replace encoded missing values with NaN
all_df['Wind Direction'] = all_df['Wind Direction'].replace('-', np.nan)
all_df['Wind Speed (km/h)'] = all_df['Wind Speed (km/h)'].replace(-9999.0, np.nan)
all_df['MSLP (hPa)'] = all_df['MSLP (hPa)'].replace(-9999.0, np.nan)

# Fill missing values
all_df['Wind Speed (km/h)'] = pd.to_numeric(all_df['Wind Speed (km/h)'], errors='coerce').interpolate(method='linear')
all_df['MSLP (hPa)']        = pd.to_numeric(all_df['MSLP (hPa)'], errors='coerce').interpolate(method='linear')
all_df['Rainfall since 9 am (mm)'] = pd.to_numeric(all_df['Rainfall since 9 am (mm)'], errors='coerce').ffill()
all_df['Wind Direction'] = all_df['Wind Direction'].ffill()

print(all_df.isna().sum())

In [ ]:
# Encode wind direction as circular features
direction_degrees = {
    'N': 0,   'NNE': 22.5, 'NE': 45,  'ENE': 67.5,
    'E': 90,  'ESE': 112.5,'SE': 135, 'SSE': 157.5,
    'S': 180, 'SSW': 202.5,'SW': 225, 'WSW': 247.5,
    'W': 270, 'WNW': 292.5,'NW': 315, 'NNW': 337.5,
    'CALM': 0,
}
wind_deg = all_df['Wind Direction'].map(direction_degrees)
all_df['wind_sin'] = np.sin(np.radians(wind_deg))
all_df['wind_cos'] = np.cos(np.radians(wind_deg))

# Incremental rainfall
all_df['rain_day'] = (all_df.index - pd.Timedelta(hours=9)).normalize()
all_df['rainfall_inc'] = (
    all_df.groupby('rain_day')['Rainfall since 9 am (mm)']
    .diff().clip(lower=0)
)
first_obs = all_df['rainfall_inc'].isna()
all_df.loc[first_obs, 'rainfall_inc'] = all_df.loc[first_obs, 'Rainfall since 9 am (mm)'].clip(lower=0)

# Resample to hourly
hourly_df = all_df.resample('h').agg({
    'Air Temp (degrees C)':     'mean',
    'Dew Pt Temp (degrees C)':  'mean',
    'Humidity (%)':             'mean',
    'wind_sin':                 'mean',
    'wind_cos':                 'mean',
    'Wind Speed (km/h)':        'mean',
    'MSLP (hPa)':               'mean',
    'rainfall_inc':             'sum',
}).rename(columns={'rainfall_inc': 'Rainfall (mm)'})
hourly_df = hourly_df.dropna(subset=['Air Temp (degrees C)'])
all_df = hourly_df

print(f"Hourly rows: {all_df.shape[0]:,}   Columns: {list(all_df.columns)}")
all_df.head()

In [ ]:
# Aggregate hourly -> daily (same as teammates)
daily_df = all_df.resample('D').agg({
    'Air Temp (degrees C)':     ['mean', 'min', 'max'],
    'Dew Pt Temp (degrees C)':  'mean',
    'Humidity (%)':             'mean',
    'wind_sin':                 'mean',
    'wind_cos':                 'mean',
    'Wind Speed (km/h)':        ['mean', 'max'],
    'MSLP (hPa)':               'mean',
    'Rainfall (mm)':            'sum',
})
daily_df.columns = ['_'.join(c).strip() for c in daily_df.columns]
daily_df = daily_df.rename(columns={
    'Air Temp (degrees C)_mean': 'temp_mean',
    'Air Temp (degrees C)_min':  'temp_min',
    'Air Temp (degrees C)_max':  'temp_max',
    'Dew Pt Temp (degrees C)_mean': 'dewpt_mean',
    'Humidity (%)_mean':         'humidity_mean',
    'wind_sin_mean':             'wind_sin',
    'wind_cos_mean':             'wind_cos',
    'Wind Speed (km/h)_mean':    'wind_speed_mean',
    'Wind Speed (km/h)_max':     'wind_speed_max',
    'MSLP (hPa)_mean':          'mslp_mean',
    'Rainfall (mm)_sum':        'rainfall_total',
})
daily_df = daily_df.dropna(subset=['temp_mean'])

# Cyclical time features
doy = daily_df.index.day_of_year
daily_df['doy_sin'] = np.sin(2 * np.pi * doy / 365)
daily_df['doy_cos'] = np.cos(2 * np.pi * doy / 365)
daily_df['month'] = daily_df.index.month

print(daily_df.shape)
daily_df.head()

## Feature Selection for Clustering

We cluster on the core daily weather variables — temperature, humidity, pressure, wind, and rainfall.
Cyclical time features are excluded so clusters reflect *weather conditions*, not just seasons.

In [ ]:
cluster_features = [
    'temp_mean', 'temp_min', 'temp_max',
    'humidity_mean', 'dewpt_mean',
    'wind_speed_mean', 'wind_sin', 'wind_cos',
    'mslp_mean', 'rainfall_total'
]

X_cluster = daily_df[cluster_features].dropna()

# Standardise — KMeans is distance-based so scale matters
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print(f"Clustering on {X_scaled.shape[0]:,} days × {X_scaled.shape[1]} features")

## Finding the Optimal Number of Clusters (k)

We use two complementary methods:
- **Elbow method** — plot inertia (within-cluster sum of squares) vs k; look for the "elbow"
- **Silhouette score** — measures how well each point fits its cluster (higher = better, max 1.0)

In [ ]:
k_range = range(2, 11)
inertias   = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, marker='o', color='steelblue', linewidth=2)
axes[0].set_title('Elbow Method — Inertia vs k')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_range, silhouettes, marker='o', color='tomato', linewidth=2)
axes[1].set_title('Silhouette Score vs k')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True, alpha=0.3)

best_k = list(k_range)[silhouettes.index(max(silhouettes))]
axes[1].axvline(best_k, color='tomato', linestyle='--', alpha=0.6, label=f'Best k={best_k}')
axes[1].legend()

plt.suptitle('Choosing k for KMeans', fontsize=13)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/kmeans_01_elbow_silhouette.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Best k by silhouette: {best_k}  (score = {max(silhouettes):.3f})")

## Fit KMeans with Chosen k

We use k=4 — a good balance between interpretability and cluster quality for weather data. (Adjust if your elbow/silhouette suggest otherwise.)

In [ ]:
K = 4  # adjust if elbow/silhouette suggest a different k

km_final = KMeans(n_clusters=K, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(X_scaled)

# Attach cluster labels back to daily_df
daily_df = daily_df.loc[X_cluster.index].copy()
daily_df['cluster'] = cluster_labels

print(daily_df['cluster'].value_counts().sort_index())

## Cluster Profiles — What Does Each Bucket Represent?

We look at the mean of each weather variable per cluster to give each bucket a descriptive label.

In [ ]:
profile = daily_df.groupby('cluster')[cluster_features].mean().round(2)
profile

In [ ]:
# Heatmap of cluster profiles (z-scored so features are comparable)
profile_z = (profile - profile.mean()) / profile.std()

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(profile_z.T, annot=profile.T.values, fmt='.1f', cmap='RdYlBu_r',
            center=0, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Z-score'})
ax.set_title('KMeans Cluster Profiles (annotated with raw means)', fontsize=13)
ax.set_xlabel('Cluster')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/kmeans_02_cluster_profiles.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Assign human-readable bucket labels based on profile
# (update these after inspecting the heatmap above)
bucket_names = {
    0: 'Bucket 0',
    1: 'Bucket 1',
    2: 'Bucket 2',
    3: 'Bucket 3',
}

# After inspecting the profile heatmap, replace with descriptive names like:
# bucket_names = {
#     0: 'Hot & Dry',
#     1: 'Cool & Rainy',
#     2: 'Mild & Calm',
#     3: 'Cold & Windy',
# }

daily_df['bucket'] = daily_df['cluster'].map(bucket_names)
print(daily_df['bucket'].value_counts())

## Cluster Visualisations

In [ ]:
# PCA to 2D for visualisation
pca_viz = PCA(n_components=2, random_state=42)
X_pca = pca_viz.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#ff7f00']
for k in range(K):
    mask = cluster_labels == k
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=colors[k], label=bucket_names[k],
               alpha=0.4, s=12, edgecolors='none')

ax.set_title('KMeans Clusters (PCA 2D projection)', fontsize=13)
ax.set_xlabel(f'PC1 ({pca_viz.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca_viz.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(title='Cluster', markerscale=2)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/kmeans_03_pca_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Temperature vs Humidity coloured by cluster — intuitive interpretation
fig, ax = plt.subplots(figsize=(10, 6))
for k in range(K):
    mask = daily_df['cluster'] == k
    ax.scatter(daily_df.loc[mask, 'temp_mean'],
               daily_df.loc[mask, 'humidity_mean'],
               c=colors[k], label=bucket_names[k],
               alpha=0.35, s=10, edgecolors='none')

ax.set_xlabel('Mean Daily Temperature (°C)')
ax.set_ylabel('Mean Daily Humidity (%)')
ax.set_title('KMeans Clusters: Temperature vs Humidity', fontsize=13)
ax.legend(title='Cluster', markerscale=3)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/kmeans_04_temp_vs_humidity.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots of temperature by cluster
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Feature Distribution by Cluster', fontsize=13)

for ax, feat, title in zip(axes,
    ['temp_mean', 'humidity_mean', 'rainfall_total'],
    ['Mean Temp (°C)', 'Humidity (%)', 'Rainfall (mm)']):
    daily_df.boxplot(column=feat, by='cluster', ax=ax,
                     flierprops=dict(marker='.', markersize=2, alpha=0.3))
    ax.set_title(title)
    ax.set_xlabel('Cluster')
plt.suptitle('')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/kmeans_05_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cluster composition by month — does each cluster correspond to a season?
monthly_counts = daily_df.groupby(['month', 'cluster']).size().unstack(fill_value=0)
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(13, 5))
monthly_counts.plot(kind='bar', ax=ax, color=colors[:K], width=0.8)
ax.set_xticks(range(12))
ax.set_xticklabels(month_labels, rotation=45)
ax.set_title('Cluster Count by Month', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Number of Days')
ax.legend(title='Cluster', labels=[bucket_names[k] for k in range(K)])
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/kmeans_06_monthly_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## Using Clusters as Features for Prediction Models

The cluster assignment for each day is a compressed, interpretable summary of the overall weather condition. We can add this as a feature for our supervised models (LR, RF, AdaBoost) in two ways:
1. **One-hot encoded** cluster dummies
2. **Raw integer** cluster label (useful for tree models)

Here we export the cluster-enriched daily dataframe so teammates can import it.

In [ ]:
# One-hot encode cluster
cluster_dummies = pd.get_dummies(daily_df['cluster'], prefix='cluster')
daily_df_enriched = pd.concat([daily_df, cluster_dummies], axis=1)

# Rain tomorrow target (same threshold as teammates)
RAIN_THRESHOLD = 0.2
daily_df_enriched['rain_tomorrow'] = (
    daily_df_enriched['rainfall_total'].shift(-1) > RAIN_THRESHOLD
).astype(int)

print(f"Enriched daily df: {daily_df_enriched.shape}")
print(f"Cluster columns added: {list(cluster_dummies.columns)}")
daily_df_enriched.head()

In [ ]:
# Check if clusters have predictive signal for rain_tomorrow
rain_by_cluster = daily_df_enriched.groupby('cluster')['rain_tomorrow'].mean().round(3)
print("Rain-tomorrow rate by cluster:")
print(rain_by_cluster)

fig, ax = plt.subplots(figsize=(7, 4))
rain_by_cluster.plot(kind='bar', ax=ax, color=colors[:K], edgecolor='white')
ax.set_title('Rain Tomorrow Rate by Weather Cluster', fontsize=12)
ax.set_xlabel('Cluster')
ax.set_ylabel('Proportion of days with rain next day')
ax.set_xticklabels([bucket_names[k] for k in range(K)], rotation=20, ha='right')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/kmeans_07_rain_rate_by_cluster.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Export enriched dataframe for other notebooks to use
daily_df_enriched.to_csv('daily_with_clusters.csv')
print("Saved: daily_with_clusters.csv")
print(f"Shape: {daily_df_enriched.shape}")

## Summary

**What we did:**
1. Ran the same data cleaning pipeline as the rest of the team (raw → hourly → daily)
2. Selected core weather features and standardised them
3. Used the elbow method + silhouette score to choose k
4. Fit KMeans with k=4, grouping days into distinct *weather condition buckets*
5. Profiled each cluster to give them interpretable labels
6. Showed that clusters have real predictive signal for rain tomorrow
7. Exported `daily_with_clusters.csv` — a one-hot encoded cluster feature that can be plugged directly into the LR/RF/AdaBoost models

**Key insight:** The clusters broadly map onto weather regimes (hot/dry, cool/rainy, etc.) and carry meaningful signal about next-day rainfall — making them a useful unsupervised feature for the supervised models.